# exp114_spatial_neighbor_prior_signal_audit train

Spatial and trajectory-shape neighbor prior audit for exp099 PF/Beam/likPF train-side pseudo-tail rows.

## Contents

1. Setup and configuration
2. Input cache and variant preview
3. Fold-safe spatial neighbor prior audit
4. Metrics and artifacts

## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Validation:", get_nested(config, "validation.strategy"))
print("Seed:", get_nested(config, "validation.seed"))
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)

## 2. Input cache and variant preview


In [ ]:
from spatial_neighbor_prior_signal_audit import (
    EXP099_FEATURE_CACHE,
    find_artifact,
    parse_cluster_method,
    parse_variants,
)

feature_cache = find_artifact(
    EXP099_FEATURE_CACHE,
    get_nested(config, "data.exp099_train_feature_cache_local"),
)
cluster_method = parse_cluster_method(config)
variants = parse_variants(config)

print("Feature cache:", feature_cache)
print("Feature cache size bytes:", feature_cache.stat().st_size)
print("Cluster method:", cluster_method)
print("Variants:")
for variant in variants:
    print(" -", variant.name, "top_k=", variant.top_k, "same_typewell=", variant.require_same_typewell_group)

preview = pd.read_csv(feature_cache, nrows=5)
display(preview[["id", "well", "target", "last_known_tvt", "md_since", "pf_ancc", "beam_mean", "likpf_mean"]])

## 3. Fold-safe spatial neighbor prior audit

The audit builds query geometry from each validation well, selects neighbor source wells from train folds only, interpolates neighbor TVT drift over `md_since`, and scores prior-only plus clipped corrections against fixed exp099 candidates.

In [ ]:
from spatial_neighbor_prior_signal_audit import run_audit, to_jsonable

summary = run_audit(config=config, paths=paths)
print(json.dumps(to_jsonable({
    "rows": summary["rows"],
    "wells": summary["wells"],
    "best_candidate": summary["best_candidate"],
    "likpf_baseline": summary["likpf_baseline"],
    "delta_best_minus_likpf_rmse": summary["delta_best_minus_likpf_rmse"],
    "prior_coverage": summary["prior_coverage"],
}), indent=2, sort_keys=True))

## 4. Metrics and artifacts


In [ ]:
artifact_paths = {name: Path(path) for name, path in summary["artifacts"].items()}
for name, path in artifact_paths.items():
    print(f"{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

candidate_metrics = pd.read_csv(artifact_paths["candidate_metrics"])
signal_metrics = pd.read_csv(artifact_paths["signal_metrics"])
display(candidate_metrics.head(20))
display(signal_metrics.sort_values(["corr_true_minus_base_vs_prior_minus_base", "sign_match_rate"], ascending=False).head(20))